# Modelado MVP — Scoring de Intención de Compra

**Objetivo:** entrenar y comparar modelos de clasificación para `Revenue` a partir de los artefactos que deja `feature_enginering.ipynb`, elegir el mejor por ROC-AUC/PR-AUC, y traducir su probabilidad estimada en la lógica del motor de recomendación (cross-selling/upselling en alta propensión, incentivo de retención en baja propensión).

Pasos:

1. Baseline: regresión logística sin balancear.
2. Manejo del desbalance de clases (~85/15): `class_weight="balanced"` y SMOTE.
3. Comparación contra modelos ensemble (Random Forest / XGBoost).
4. Selección del mejor modelo por precision, recall, F1, ROC-AUC y PR-AUC (por sobre accuracy).
5. Motor de recomendación: mapear la probabilidad de compra a una acción.

## 1. Configuración y carga de artefactos

In [1]:
import sys
from pathlib import Path

import joblib
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.metrics import evaluar_modelo, comparar_modelos
from src.recommender import asignar_accion, resumen_acciones

CARPETA_MODELOS = PROJECT_ROOT / "data" / "models"

split = joblib.load(CARPETA_MODELOS / "train_test_split.joblib")
preprocessor = joblib.load(CARPETA_MODELOS / "preprocessor.joblib")

X_train, X_test = split["X_train"], split["X_test"]
y_train, y_test = split["y_train"], split["y_test"]

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}, balance: {y_train.mean():.4f}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}, balance: {y_test.mean():.4f}")

X_train: (9764, 78), y_train: (9764,), balance: 0.1563
X_test: (2441, 78), y_test: (2441,), balance: 0.1565


`X_train`/`X_test` ya están codificadas y escaladas (`preprocessor`, ajustado solo con train en `feature_enginering.ipynb`) — está todo listo para empezar a entrenar.

## 2. Baseline: Regresión Logística sin balancear

In [2]:
resultados = {}
modelos = {}

logreg_base = LogisticRegression(max_iter=1000, random_state=42)
logreg_base.fit(X_train, y_train)

y_pred = logreg_base.predict(X_test)
y_prob = logreg_base.predict_proba(X_test)[:, 1]

resultados["logreg_baseline"] = evaluar_modelo(y_test, y_pred, y_prob)
modelos["logreg_baseline"] = logreg_base
resultados["logreg_baseline"]

{'precision': 0.7655502392344498,
 'recall': 0.418848167539267,
 'f1': 0.5414551607445008,
 'roc_auc': 0.9009367633858758,
 'average_precision': 0.6542789391437379}

Entrenada tal cual sobre el train desbalanceado, la regresión logística tiende a predecir la clase mayoritaria (`Revenue=0`): el recall de la clase positiva queda bajo, porque el modelo minimiza el error global y el 85% de los casos ya son `No compra`. Esto motiva las estrategias de balanceo de la siguiente sección.

## 3. Manejo del desbalance de clases

### 3.1 `class_weight="balanced"`

In [3]:
logreg_cw = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
logreg_cw.fit(X_train, y_train)

y_pred = logreg_cw.predict(X_test)
y_prob = logreg_cw.predict_proba(X_test)[:, 1]

resultados["logreg_class_weight"] = evaluar_modelo(y_test, y_pred, y_prob)
modelos["logreg_class_weight"] = logreg_cw
resultados["logreg_class_weight"]

{'precision': 0.5032679738562091,
 'recall': 0.806282722513089,
 'f1': 0.6197183098591549,
 'roc_auc': 0.9111867449506572,
 'average_precision': 0.6638926292293772}

`class_weight="balanced"` penaliza más los errores sobre la clase minoritaria en la función de pérdida, sin tocar los datos ni el conjunto de test.

### 3.2 SMOTE

In [4]:
X_train_smote, y_train_smote = SMOTE(random_state=42).fit_resample(X_train, y_train)
print(f"Train original: {X_train.shape[0]} filas, balance: {y_train.mean():.4f}")
print(f"Train con SMOTE: {X_train_smote.shape[0]} filas, balance: {y_train_smote.mean():.4f}")

logreg_smote = LogisticRegression(max_iter=1000, random_state=42)
logreg_smote.fit(X_train_smote, y_train_smote)

y_pred = logreg_smote.predict(X_test)
y_prob = logreg_smote.predict_proba(X_test)[:, 1]

resultados["logreg_smote"] = evaluar_modelo(y_test, y_pred, y_prob)
modelos["logreg_smote"] = logreg_smote
resultados["logreg_smote"]

Train original: 9764 filas, balance: 0.1563
Train con SMOTE: 16476 filas, balance: 0.5000


{'precision': 0.502495840266223,
 'recall': 0.7905759162303665,
 'f1': 0.6144455747711088,
 'roc_auc': 0.9100819032265447,
 'average_precision': 0.6649388726510547}

SMOTE genera ejemplos sintéticos de la clase minoritaria por interpolación entre vecinos, y se aplica **solo sobre `X_train`**: `X_test` nunca se toca, para que la evaluación siga midiendo sobre datos reales y no se filtre información sintética al conjunto de test.

In [5]:
comparar_modelos(resultados)

,precision,recall,f1,roc_auc,average_precision
logreg_class_weight,0.503268,0.806283,0.619718,0.911187,0.663893
logreg_smote,0.502496,0.790576,0.614446,0.910082,0.664939
logreg_baseline,0.765550,0.418848,0.541455,0.900937,0.654279


## 4. Modelos ensemble

### 4.1 Random Forest

In [6]:
rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

resultados["random_forest"] = evaluar_modelo(y_test, y_pred, y_prob)
modelos["random_forest"] = rf
resultados["random_forest"]

{'precision': 0.6577669902912622,
 'recall': 0.7094240837696335,
 'f1': 0.6826196473551638,
 'roc_auc': 0.924264943333952,
 'average_precision': 0.7226931082367187}

### 4.2 XGBoost

In [7]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
)
xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:, 1]

resultados["xgboost"] = evaluar_modelo(y_test, y_pred, y_prob)
modelos["xgboost"] = xgb
resultados["xgboost"]

{'precision': 0.6458852867830424,
 'recall': 0.6780104712041884,
 'f1': 0.6615581098339719,
 'roc_auc': 0.9220641596464507,
 'average_precision': 0.7170867562321463}

`scale_pos_weight` es el equivalente de `class_weight="balanced"` para XGBoost: la razón entre negativos y positivos en train, para que el árbol le dé más peso a los errores sobre la clase minoritaria.

## 5. Comparación de modelos

In [8]:
tabla_comparativa = comparar_modelos(resultados)
tabla_comparativa

,precision,recall,f1,roc_auc,average_precision
random_forest,0.657767,0.709424,0.682620,0.924265,0.722693
xgboost,0.645885,0.678010,0.661558,0.922064,0.717087
logreg_class_weight,0.503268,0.806283,0.619718,0.911187,0.663893
logreg_smote,0.502496,0.790576,0.614446,0.910082,0.664939
logreg_baseline,0.765550,0.418848,0.541455,0.900937,0.654279


In [9]:
mejor_modelo_nombre = tabla_comparativa.index[0]
mejor_modelo = modelos[mejor_modelo_nombre]
print(f"Mejor modelo por ROC-AUC: {mejor_modelo_nombre}")

Mejor modelo por ROC-AUC: random_forest


Se elige por `roc_auc` (capacidad general de discriminar entre clases) y se mira `average_precision` como segundo criterio, más informativo que ROC-AUC cuando la clase positiva es minoritaria.

## 6. Motor de recomendación

In [10]:
y_prob_final = mejor_modelo.predict_proba(X_test)[:, 1]
acciones = asignar_accion(y_prob_final)
acciones.value_counts()

accion
retencion        1794
sin_accion        422
cross_selling     225
Name: count, dtype: int64

In [11]:
resumen_acciones(acciones, y_test)

,n_sesiones,tasa_compra_real,pct_del_total
accion,,,
retencion,1794,0.026756,73.49
sin_accion,422,0.393365,17.29
cross_selling,225,0.746667,9.22


Las sesiones con `cross_selling` (probabilidad > 70%) deberían concentrar la mayor tasa real de compra, y las de `retencion` (< 30%) la menor — si no es así, conviene revisar los umbrales antes de activar las acciones en producción.

## 7. Persistencia del modelo final

In [12]:
ruta_modelo = CARPETA_MODELOS / "modelo_final.joblib"
joblib.dump(mejor_modelo, ruta_modelo)
print(f"Modelo final ({mejor_modelo_nombre}) guardado en: {ruta_modelo}")

Modelo final (random_forest) guardado en: /home/juanma/HENRRY/PROYECTO_FINAL/Proyecto-Final-Henry/data/models/modelo_final.joblib


## 8. Conclusiones

- El baseline sin balancear confirma el problema: buen desempeño global, mal recall en `Revenue=1`.
- `class_weight="balanced"` y SMOTE mejoran el recall de la clase positiva frente al baseline, a costa de algo de precision.
- Los modelos ensemble (Random Forest, XGBoost) se comparan contra las variantes de regresión logística bajo las mismas métricas.
- El modelo final se elige por ROC-AUC/PR-AUC y queda persistido en `data/models/modelo_final.joblib`, listo para servir predicciones.
- El motor de recomendación traduce la probabilidad en una acción concreta (`cross_selling`, `retencion`, `sin_accion`), auditada contra la compra real.